# Kokoro-82M — DIMER text-to-speech tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/kokoro-tts-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/kokoro-tts-pipeline/blob/main/tutorials/kokoro_tts_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-hexgrad%2FKokoro--82M-ffcc4d?style=flat)](https://huggingface.co/hexgrad/Kokoro-82M) [![Upstream](https://img.shields.io/badge/Upstream-hexgrad%2Fkokoro-181717?style=flat&logo=github&logoColor=white)](https://github.com/hexgrad/kokoro) [![arXiv](https://img.shields.io/badge/arXiv-2306.07691-b31b1b.svg)](https://arxiv.org/abs/2306.07691)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** text-to-speech (24 kHz mono float32 waveform from a named synthetic voice pack) using the pinned Kokoro-82M v1.0 weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/kokoro_tts_pipeline/pipeline.py` at revision `943077dd6731`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `f3ff3571791e39611d31c381e3a41a3af07b4987` (~355 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the `misaki` grapheme-to-phoneme library turns the text into a phoneme string, a 128-d style vector is read from the selected pre-computed voice pack (indexed by phoneme count), the StyleTTS 2 decoder predicts per-phoneme durations, pitch and energy, and the ISTFTNet vocoder renders one 24 kHz mono waveform per text segment; the segments are concatenated. **No adaptation occurs:** no training, fine-tuning, voice cloning, in-context conditioning, or preprocessing fitting — the only choice is which of the 54 shipped voice packs to use. What the upstream snapshot supplies is the checkpoint, the configuration and the voice packs; what the carried pipeline module adds is manifest staging and SHA-256 verification of all 58 snapshot files, input validation and ceilings, one-language-per-instance voice checking, a fixed output contract and the `validate_inputs` and `evaluation_report` stage helpers. **Speech quality has no intrinsic metric:** the pipeline ships no metric helper because naturalness (MOS) needs human listeners and intelligibility (ASR word error rate) needs an external recogniser; the notebook reports run-level facts (duration, peak amplitude, phonemes) and no quality score.

**Trust boundary:** the checkpoint and the voice packs are PyTorch pickle files, not SafeTensors (`WEIGHT_FORMAT`). In Section 3 digest verification runs before any file is opened, and the `kokoro` library then deserialises them with the weights-only loader (`LOADER_WEIGHTS_ONLY`), which restricts unpickling to tensors and primitive containers. Path-safety and digest checks do not make an unverified pickle safe; only the pinned, verified bytes ever reach the unpickler, and there is no fallback to a different download. `from_pretrained` then loads from that verified directory with `lang_code='a'` and reports `espeak_fallback`: whether the espeak-ng fallback for out-of-dictionary words bound on this host (the pinned `espeakng-loader` wheel bundles the library).

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, author a synthetic English sentence (or upload your own text), stage and digest-verify the immutable upstream snapshot including every voice pack, surface the pipeline's ceilings and voice/language rules and validate the request into an input manifest, synthesise speech through the public API with an explicit seed, read the output contract correctly, write a playable WAV under `outputs/`, read from the machine-readable evaluation report why no metric is reported and what external judges a real evaluation needs, and export machine-readable results plus provenance.

**This notebook does not demonstrate:** voice cloning or speaker adaptation (Kokoro has no speaker encoder and accepts no reference audio), speech-to-text (the `whisper-asr-pipeline` sibling covers that), voice mixing, SSML or emotion control, word-level timestamps, languages other than the pipeline's `lang_code` at load time, or any quality score. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (also float32; the pipeline does not change precision by device). The model card's CPU smoke loaded and verified the 58-file snapshot in 6.22 s and rendered 3.25 s of audio in 0.72 s, so the one-sentence default runs in seconds on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 355 MB snapshot (327 MB checkpoint plus 54 voice packs) are the largest downloads of the run.
- **Knowledge:** basic Python; what a phoneme string is; why a synthesised waveform has no ground truth to score against.
- **Data:** the default sample is one synthetic English sentence authored in code, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one UTF-8 text file of at most 2,000 characters; the pipeline splits it on newlines and synthesises each line as a segment. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API. The text you submit will be spoken verbatim.
- **Dependency network call:** outside this package's control, the `misaki` G2P library downloads the spaCy `en_core_web_sm` model once on first English use.
- **External access:** the Hugging Face Hub only, to fetch the pinned `hexgrad/Kokoro-82M` snapshot (~355 MB in total) at revision `f3ff3571791e…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `kokoro`, `misaki`, `soundfile` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'kokoro==0.9.4',
    'misaki==0.9.4',
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'numpy==2.5.3',
    'huggingface-hub==0.36.2',
    'soundfile==0.14.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'kokoro-tts-pipeline',
    'repository_revision': '943077dd6731f6974e5b65a86d57a31f2efc074a',
    'embedded_module': 'src/kokoro_tts_pipeline/pipeline.py',
    'embedded_modules': ['src/kokoro_tts_pipeline/pipeline.py'],
    'module_sha256': '7c3c454cee940e29f3e701816b49901e380c3337991532348bce382fd10a2f83',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, kokoro, misaki, soundfile
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'kokoro': kokoro.__version__, 'misaki': misaki.__version__, 'soundfile': soundfile.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/kokoro_tts_pipeline/` @ `943077dd6731`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/kokoro_tts_pipeline/pipeline.py`

In [ ]:
"""Text-to-speech with the pinned ``hexgrad/Kokoro-82M`` snapshot (24 kHz mono float32).

The checkpoint ``kokoro-v1_0.pth`` and every ``voices/*.pt`` pack are PyTorch pickle files, not SafeTensors.
The trust boundary is therefore: (1) every file is SHA-256-verified against the manifest before it is opened,
and (2) the ``kokoro`` library deserialises both with ``torch.load`` under ``weights_only=True`` (recorded
in ``LOADER_WEIGHTS_ONLY``), which restricts unpickling to tensors and primitive containers.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "hexgrad/Kokoro-82M"
MODEL_REVISION = "f3ff3571791e39611d31c381e3a41a3af07b4987"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "kokoro-82m"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "kokoro-v1_0.pth"
CONFIG_FILE = "config.json"
VOICES_DIR = "voices"

WEIGHT_FORMAT = "pytorch-pickle"  # .pth checkpoint and .pt voice packs; not SafeTensors
# kokoro 0.9.4 model.py:68 and pipeline.py:147 call torch.load(..., weights_only=True)
LOADER_WEIGHTS_ONLY = True
SAMPLE_RATE = 24000  # Hz, mono, float32 (upstream README)
LANG_CODES = ("a", "b", "e", "f", "h", "i", "j", "p", "z")  # kokoro.pipeline.LANG_CODES keys
DEFAULT_LANG_CODE = "a"  # American English
DEFAULT_VOICE = "af_heart"
MAX_TEXT_CHARS = 2000  # characters per synthesize() call; the library chunks at 510 phonemes per segment
MIN_SPEED = 0.5
MAX_SPEED = 2.0


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def list_voices(manifest: dict[str, Any]) -> tuple[str, ...]:
    """Voice names (``af_heart`` …) from the manifest's ``voices/*.pt`` entries, sorted."""
    prefix = f"{VOICES_DIR}/"
    paths = [entry["path"] for entry in manifest["files"]]
    names = [path for path in paths if path.startswith(prefix) and path.endswith(".pt")]
    return tuple(sorted(name[len(prefix) : -3] for name in names))


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one non-empty str, spoken verbatim; newline-split into segments by the library",
    "text_chars": [1, MAX_TEXT_CHARS],
    "speed": [MIN_SPEED, MAX_SPEED],
    "voice": (
        "one of the manifest voices/*.pt packs whose first letter equals the instance "
        f"lang_code ({DEFAULT_LANG_CODE} by default)"
    ),
    "lang_codes": list(LANG_CODES),
    "sample_rate_hz": SAMPLE_RATE,
    "preprocessing": (
        "grapheme-to-phoneme through misaki (espeak-ng fallback for out-of-dictionary words when it "
        "binds on the host), a 128-d style vector read from the voice pack by phoneme count, and a "
        "510-phoneme cap per segment applied inside the library"
    ),
}


def _check_inputs(text: Any, voice: Any, speed: Any, voices: Sequence[str], lang_code: str) -> None:
    """Raise TypeError/ValueError naming the first violated ceiling; return nothing."""
    if not isinstance(text, str):
        raise TypeError("text must be a str")
    if not text.strip():
        raise ValueError("text must not be empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"text exceeds MAX_TEXT_CHARS={MAX_TEXT_CHARS}: {len(text)}")
    if not isinstance(voice, str) or voice not in voices:
        raise ValueError(f"voice must be one of the {len(voices)} manifest voices, got {voice!r}")
    if voice[0] != lang_code:
        raise ValueError(f"voice {voice!r} is not a lang_code={lang_code!r} voice")
    if isinstance(speed, bool) or not isinstance(speed, int | float):
        raise TypeError("speed must be a number")
    if not MIN_SPEED <= speed <= MAX_SPEED:
        raise ValueError(f"speed must be between MIN_SPEED={MIN_SPEED} and MAX_SPEED={MAX_SPEED}")


def validate_inputs(
    text: str,
    voice: str = DEFAULT_VOICE,
    *,
    speed: float = 1.0,
    voices: Sequence[str],
    lang_code: str = DEFAULT_LANG_CODE,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    ``voices`` is the instance's authoritative voice inventory — ``pipe.voices``, which
    ``list_voices`` derives from the digest-verified manifest — and is required because voice
    membership cannot be checked without it. Rejection is reported by raising exactly as
    ``synthesize`` would: both route through ``_check_inputs``.
    """
    _check_inputs(text, voice, speed, voices, lang_code)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry: synthesize takes one text per call")
    lines = [line for line in text.splitlines() if line.strip()]
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "text-0",
                "chars": len(text),
                "segments": len(lines) or 1,
            }
        ],
        "voice": voice,
        "speed": float(speed),
        "lang_code": lang_code,
        "voice_inventory": len(voices),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], references: Sequence[Any] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    Speech quality has no intrinsic metric and the repository ships no metric helper, so the
    verdict is always ``not-measurable`` (EVAL9). ``references`` exists for interface parity with
    the fleet's other pipelines and is recorded in ``reason`` rather than scored: a reference
    recording cannot be compared to a synthesised waveform sample-by-sample, and both of the real
    judges (listener MOS, ASR word error rate) live outside this repository.
    """
    supplied = references is not None
    return {
        "task": f"text-to-speech synthesis ({SAMPLE_RATE} Hz mono float32, named synthetic voice pack)",
        "score_semantics": (
            "the output is a waveform, not a prediction: duration, peak amplitude and the phoneme "
            "string are run-level facts, not quality scores, and none of them bounds naturalness "
            "or intelligibility"
        ),
        "sample_kind": sample_kind,
        "n_segments": len(result.get("segments", [])),
        "duration_s": float(result.get("duration_s", 0.0)),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "speech quality has no ground truth in this repository and no metric helper is shipped"
            + (
                "; references were supplied but no metric helper exists to score them here"
                if supplied
                else "; the evaluated sample has no reference recording"
            )
        ),
        "needs": (
            "an external judge: Mean Opinion Score ratings from human listeners for naturalness, or "
            "an independent speech recogniser to re-transcribe the waveform and compute word error "
            "rate against the input text for intelligibility — over a reference sentence set, with "
            "the judge named"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class KokoroTTSPipeline:
    """``_runner(text, voice_path, speed)`` yields ``(graphemes, phonemes, audio_1d_float32)`` per segment."""

    _runner: Callable[..., Any]
    voices: tuple[str, ...]
    lang_code: str = DEFAULT_LANG_CODE
    device: str = "cpu"
    source: str = "injected"
    weights_dir: Path = DEFAULT_WEIGHTS_DIR
    espeak_fallback: bool | None = None  # None = unknown (injected runner)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        lang_code: str = DEFAULT_LANG_CODE,
    ) -> KokoroTTSPipeline:
        if lang_code not in LANG_CODES:
            raise ValueError(f"lang_code must be one of LANG_CODES {LANG_CODES}, got {lang_code!r}")
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(
                f"no verified snapshot at {root} and no Hub path is offered for pickle checkpoints; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        stage_missing_files(root, allow_download=allow_download)
        manifest = verify_snapshot(root)
        # Validate language and verify the snapshot before importing model libraries.
        import torch
        from kokoro import KModel, KPipeline

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        kmodel = KModel(repo_id=MODEL_ID, config=str(root / CONFIG_FILE), model=str(root / WEIGHTS_FILE))
        kmodel = kmodel.to(resolved_device).eval()
        kpipeline = KPipeline(lang_code=lang_code, repo_id=MODEL_ID, model=kmodel, device=resolved_device)
        fallback = getattr(getattr(kpipeline, "g2p", None), "fallback", None)

        def runner(text: str, voice_path: str, speed: float) -> Any:
            with torch.inference_mode():
                for result in kpipeline(text, voice=voice_path, speed=speed, split_pattern=r"\n+"):
                    audio = result.audio
                    array = audio.cpu().numpy() if audio is not None else None
                    yield result.graphemes, result.phonemes, array

        return cls(
            runner, list_voices(manifest), lang_code, resolved_device, "local-snapshot", root,
            fallback is not None if lang_code in "ab" else None,
        )

    def _validate(self, text: Any, voice: Any, speed: Any) -> None:
        _check_inputs(text, voice, speed, self.voices, self.lang_code)

    def synthesize(self, text: str, voice: str = DEFAULT_VOICE, *, speed: float = 1.0) -> dict[str, Any]:
        """Synthesise ``text`` with a named voice pack; ``audio`` is a 1-D float32 array at 24 kHz."""
        self._validate(text, voice, speed)
        voice_path = str(self.weights_dir / VOICES_DIR / f"{voice}.pt")
        chunks: list[np.ndarray] = []
        segments: list[dict[str, str]] = []
        for graphemes, phonemes, audio in self._runner(text, voice_path, float(speed)):
            if audio is None:
                continue
            array = np.asarray(audio, dtype=np.float32).reshape(-1)
            chunks.append(array)
            segments.append({"graphemes": str(graphemes), "phonemes": str(phonemes)})
        if not chunks:
            raise RuntimeError("no audio produced: every segment was empty after grapheme-to-phoneme")
        wave = np.concatenate(chunks)
        return {
            "audio": wave,
            "sample_rate": SAMPLE_RATE,
            "num_samples": int(wave.shape[0]),
            "duration_s": float(wave.shape[0] / SAMPLE_RATE),
            "peak_amplitude": float(np.abs(wave).max()),
            "segments": segments,
            "voice": voice,
            "lang_code": self.lang_code,
            "speed": float(speed),
            "espeak_fallback": self.espeak_fallback,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `58`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `f3ff3571791e…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `KokoroTTSPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "kokoro-82m",
  "modelId": "hexgrad/Kokoro-82M",
  "revision": "f3ff3571791e39611d31c381e3a41a3af07b4987",
  "files": [
    {
      "path": "README.md",
      "bytes": 6348,
      "sha256": "91dcabced89db6f109b8786642f50402d3ee87450e8189589b6f85520e7f4d78"
    },
    {
      "path": "VOICES.md",
      "bytes": 7625,
      "sha256": "ec7e4941ad7e194af61e3455928528a9ff5360c7c505e412efab27d6a69ea106"
    },
    {
      "path": "config.json",
      "bytes": 2351,
      "sha256": "5abb01e2403b072bf03d04fde160443e209d7a0dad49a423be15196b9b43c17f"
    },
    {
      "path": "kokoro-v1_0.pth",
      "bytes": 327212226,
      "sha256": "496dba118d1a58f5f3db2efc88dbdc216e0483fc89fe6e47ee1f2c53f18ad1e4"
    },
    {
      "path": "voices/af_alloy.pt",
      "bytes": 523425,
      "sha256": "6d877149dd8b348fbad12e5845b7e43d975390e9f3b68a811d1d86168bef5aa3"
    },
    {
      "path": "voices/af_aoede.pt",
      "bytes": 523425,
      "sha256": "c03bd1a4c3716c2d8eaa3d50022f62d5c31cfbd6e15933a00b17fefe13841cc4"
    },
    {
      "path": "voices/af_bella.pt",
      "bytes": 523425,
      "sha256": "8cb64e02fcc8de0327a8e13817e49c76c945ecf0052ceac97d3081480e8e48d6"
    },
    {
      "path": "voices/af_heart.pt",
      "bytes": 523425,
      "sha256": "0ab5709b8ffab19bfd849cd11d98f75b60af7733253ad0d67b12382a102cb4ff"
    },
    {
      "path": "voices/af_jessica.pt",
      "bytes": 523435,
      "sha256": "cdfdccb8cc975aa34ee6b89642963b0064237675de0e41a30ae64cc958dd4e87"
    },
    {
      "path": "voices/af_kore.pt",
      "bytes": 523420,
      "sha256": "8bfbc512321c3db49dff984ac675fa5ac7eaed5a96cc31104d3a9080e179d69d"
    },
    {
      "path": "voices/af_nicole.pt",
      "bytes": 523430,
      "sha256": "c5561808bcf5250fe8c5f5de32caf2d94f27e57e95befdb098c5c85991d4c5da"
    },
    {
      "path": "voices/af_nova.pt",
      "bytes": 523420,
      "sha256": "e0233676ddc21908c37a1f102f6b88a59e4e5c1bd764983616eb9eda629dbcd2"
    },
    {
      "path": "voices/af_river.pt",
      "bytes": 523425,
      "sha256": "e149459bd9c084416b74756b9bd3418256a8b839088abb07d463730c369dab8f"
    },
    {
      "path": "voices/af_sarah.pt",
      "bytes": 523425,
      "sha256": "49bd364ea3be9eb3e9685e8f9a15448c4883112a7c0ff7ab139fa4088b08cef9"
    },
    {
      "path": "voices/af_sky.pt",
      "bytes": 523351,
      "sha256": "c799548aed06e0cb0d655a85a01b48e7f10484d71663f9a3045a5b9362e8512c"
    },
    {
      "path": "voices/am_adam.pt",
      "bytes": 523420,
      "sha256": "ced7e284aba12472891be1da3ab34db84cc05cc02b5889535796dbf2d8b0cb34"
    },
    {
      "path": "voices/am_echo.pt",
      "bytes": 523420,
      "sha256": "8bcfdc852bc985fb45c396c561e571ffb9183930071f962f1b50df5c97b161e8"
    },
    {
      "path": "voices/am_eric.pt",
      "bytes": 523420,
      "sha256": "ada66f0eefff34ec921b1d7474d7ac8bec00cd863c170f1c534916e9b8212aae"
    },
    {
      "path": "voices/am_fenrir.pt",
      "bytes": 523430,
      "sha256": "98e507eca1db08230ae3b6232d59c10aec9630022d19accac4f5d12fcec3c37a"
    },
    {
      "path": "voices/am_liam.pt",
      "bytes": 523420,
      "sha256": "c82550757ddb31308b97f30040dda8c2d609a9e2de6135848d0a948368138518"
    },
    {
      "path": "voices/am_michael.pt",
      "bytes": 523435,
      "sha256": "9a443b79a4b22489a5b0ab7c651a0bcd1a30bef675c28333f06971abbd47bd37"
    },
    {
      "path": "voices/am_onyx.pt",
      "bytes": 523420,
      "sha256": "e8452be16cd0f6da7b4579eaf7b1e4506e92524882053d86d72b96b9a7fed584"
    },
    {
      "path": "voices/am_puck.pt",
      "bytes": 523420,
      "sha256": "dd1d8973f4ce4b7d8ae407c77a435f485dabc052081b80ea75c4f30b84f36223"
    },
    {
      "path": "voices/am_santa.pt",
      "bytes": 523425,
      "sha256": "7f2f7582fa2b1f160e90aafe6d0b442a685e773608b6667e545d743b073e97a7"
    },
    {
      "path": "voices/bf_alice.pt",
      "bytes": 523425,
      "sha256": "d292651b6af6c0d81705c2580dcb4463fccc0ff7b8d618a471dbb4e45655b3f3"
    },
    {
      "path": "voices/bf_emma.pt",
      "bytes": 523420,
      "sha256": "d0a423deabf4a52b4f49318c51742c54e21bb89bbbe9a12141e7758ddb5da701"
    },
    {
      "path": "voices/bf_isabella.pt",
      "bytes": 523440,
      "sha256": "cdd4c37003805104d1d08fb1e05855c8fb2c68de24ca6e71f264a30aaa59eefd"
    },
    {
      "path": "voices/bf_lily.pt",
      "bytes": 523420,
      "sha256": "6e09c2e481e2d53004d7e5ae7d3a325369e130a6f45c35a6002de75084be9285"
    },
    {
      "path": "voices/bm_daniel.pt",
      "bytes": 523430,
      "sha256": "fc3fce4e9c12ed4dbc8fa9680cfe51ee190a96444ce7c3ad647549a30823fc5d"
    },
    {
      "path": "voices/bm_fable.pt",
      "bytes": 523425,
      "sha256": "d44935f3135257a9064df99f007fc1342ff1aa767552b4a4fa4c3b2e6e59079c"
    },
    {
      "path": "voices/bm_george.pt",
      "bytes": 523430,
      "sha256": "f1bc812213dc59774769e5c80004b13eeb79bd78130b11b2d7f934542dab811b"
    },
    {
      "path": "voices/bm_lewis.pt",
      "bytes": 523425,
      "sha256": "b5204750dcba01029d2ac9cec17aec3b20a6d64073c579d694a23cb40effbd0e"
    },
    {
      "path": "voices/ef_dora.pt",
      "bytes": 523420,
      "sha256": "d9d69b0f8a2b87a345f269d89639f89dfbd1a6c9da0c498ae36dd34afcf35530"
    },
    {
      "path": "voices/em_alex.pt",
      "bytes": 523420,
      "sha256": "5eac53f767c3f31a081918ba531969aea850bed18fe56419b804d642c6973431"
    },
    {
      "path": "voices/em_santa.pt",
      "bytes": 523430,
      "sha256": "aa8620cb96cec705823efca0d956a63e158e09ad41aca934d354b7f0778f63cb"
    },
    {
      "path": "voices/ff_siwis.pt",
      "bytes": 523425,
      "sha256": "8073bf2d2c4b9543a90f2f0fd2144de4ed157e2d4b79ddeb0d5123066171fbc9"
    },
    {
      "path": "voices/hf_alpha.pt",
      "bytes": 523425,
      "sha256": "06906fe05746d13a79c5c01e21fd7233b05027221a933c9ada650f5aafc8f044"
    },
    {
      "path": "voices/hf_beta.pt",
      "bytes": 523420,
      "sha256": "63c0a1a6272e98d43f4511bba40e30dd9c8ceaf5f39af869509b9f51a71c503e"
    },
    {
      "path": "voices/hm_omega.pt",
      "bytes": 523425,
      "sha256": "b55f02a8e8483fffe0afa566e7d22ed8013acf47ad4f6bbee2795a840155703e"
    },
    {
      "path": "voices/hm_psi.pt",
      "bytes": 523351,
      "sha256": "2f0f055cea4f1083f4ef127ece48d71606347f6557dbb961c0ca5740a2da485b"
    },
    {
      "path": "voices/if_sara.pt",
      "bytes": 523425,
      "sha256": "6c0b253b955fe32f1a1a86006aebe83d050ea95afd0e7be15182f087deedbf55"
    },
    {
      "path": "voices/im_nicola.pt",
      "bytes": 523341,
      "sha256": "234ed06648649f9bd874b37508ea17560b9c993ef85b4ddb3e3a71e062bd2c12"
    },
    {
      "path": "voices/jf_alpha.pt",
      "bytes": 523425,
      "sha256": "1bf4c9dc69e45ee46183b071f4db766349aac5592acbcfeaf051018048a5d787"
    },
    {
      "path": "voices/jf_gongitsune.pt",
      "bytes": 523351,
      "sha256": "1b171917f18f351e65f2bf9657700cd6bfec4e65589c297525b9cf3c20105770"
    },
    {
      "path": "voices/jf_nezumi.pt",
      "bytes": 523420,
      "sha256": "d83f007a7f01783b77014561a7d493d327a0210e143440e91c9b697590d27661"
    },
    {
      "path": "voices/jf_tebukuro.pt",
      "bytes": 523435,
      "sha256": "0d6917904438aec85f73a6fa1f7ac2be6481aae47c697834936930a91796c576"
    },
    {
      "path": "voices/jm_kumo.pt",
      "bytes": 523425,
      "sha256": "98340afd68b1cee84fe0cd95528cfa6d4b39e416aa75a9df64049d52c8b55896"
    },
    {
      "path": "voices/pf_dora.pt",
      "bytes": 523425,
      "sha256": "07e4ff987c5d5a8c3995efd15cc4f0db7c4c15e881b198d8ab7f67ecf51f5eb7"
    },
    {
      "path": "voices/pm_alex.pt",
      "bytes": 523425,
      "sha256": "cf0ba8c573c2480fc54123683a35cf1e2ae130428e441eb91f9149bdb188a526"
    },
    {
      "path": "voices/pm_santa.pt",
      "bytes": 523430,
      "sha256": "d42103169c5c872abbafb9129133af7e942bb9d272c3cc3b95c203e7d7198c29"
    },
    {
      "path": "voices/zf_xiaobei.pt",
      "bytes": 523435,
      "sha256": "9b76be63dab4f4f96962030acc0126a9aee9728608fbbe115e2b58a2bd504df6"
    },
    {
      "path": "voices/zf_xiaoni.pt",
      "bytes": 523430,
      "sha256": "95b49f169bf1640f4f43c25e13daa39f7b98d15d00823e83ef5c14b3ced59e49"
    },
    {
      "path": "voices/zf_xiaoxiao.pt",
      "bytes": 523440,
      "sha256": "cfaf6f2ded1ee56f1ff94fcd2b0e6cdf32e5b794bdc05b44e7439d44aef5887c"
    },
    {
      "path": "voices/zf_xiaoyi.pt",
      "bytes": 523430,
      "sha256": "b5235dbaeef85a4c613bf78af9a88ff63c25bac5f26ba77e36186d8b7ebf05e2"
    },
    {
      "path": "voices/zm_yunjian.pt",
      "bytes": 523435,
      "sha256": "76cbf8bad35901d011d9628a2fdceb7b4f1f127e7a3269cb393b3941eb7fc417"
    },
    {
      "path": "voices/zm_yunxi.pt",
      "bytes": 523425,
      "sha256": "dbe6e1ce7c3dbaf2f5667432947b638b1c6831ccbe154c4610dbcc44f431e27b"
    },
    {
      "path": "voices/zm_yunxia.pt",
      "bytes": 523430,
      "sha256": "bb2b03b08e84d64e1214440ce3b624987fac177f2eeb5bab8571799a3d980acd"
    },
    {
      "path": "voices/zm_yunyang.pt",
      "bytes": 523435,
      "sha256": "5238ac22e0c7f8b6cdd2eddd6e444b8a700b73c4674d9a047d59a94ff96379a2"
    }
  ],
  "totalBytes": 355493259
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = KokoroTTSPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Author the sample text or optional BYOD

The default sample is **synthetic**: one English pangram written in this cell — the same sentence the model card's CPU smoke used, chosen because every word is in the `misaki` dictionary, so the espeak-ng fallback is not needed to pronounce it. It exists to prove the code path, not to measure anything: it ships **no reference recording**, so the audio it produces is smoke/sanity evidence that the pipeline works, never a quality measurement and never benchmark evidence. The voice, the speed and the seed are Colab form parameters so they can be changed without editing code; their allowed ranges are checked against the carried module in Section 5.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file of at most `MAX_TEXT_CHARS` characters; the pipeline splits it on newlines and synthesises each line as a segment, and the library truncates any single segment above 510 phonemes with a warning, so keep lines to a sentence or two. The upload stays inside this runtime. Numbers, acronyms and names outside the dictionary are pronounced by the espeak-ng fallback when it is available on the host and are otherwise dropped — Section 6 shows how to check.

In [ ]:
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}
VOICE = 'af_heart'  # @param {type:"string"}
SPEED = 1.0  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    text = io.TextIOWrapper(io.BytesIO(uploaded[sample_name]), encoding='utf-8').read().strip()
    sample_kind = 'BYOD upload'
else:
    text = 'The quick brown fox jumps over the lazy dog.'
    sample_name = 'synthetic_pangram'
    sample_kind = 'synthetic (authored in this cell; the model card smoke sentence)'
text_sha256 = hashlib.sha256(text.encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'chars': len(text), 'lines': len(text.splitlines()), 'text_sha256': text_sha256, 'voice': VOICE, 'speed': SPEED, 'seed': SEED})
print(text[:200])

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `synthesize` applies — both route through the same private `_check_inputs` — so the text type, non-emptiness, the character ceiling `MAX_TEXT_CHARS`, voice membership in the digest-verified voice inventory, the one-language-per-instance rule and the `MIN_SPEED`–`MAX_SPEED` range are enforced identically. The voice inventory is passed in as `voices=pipe.voices`, which `list_voices` read from the verified manifest in Section 3 — the only authoritative voice list. The helper returns an **input manifest** naming the schema and ceilings, the request's character count and segment count, the voice, speed and language code in force, and the verdict; it is written to `outputs/kokoro_tts_input_manifest.json`. `LANG_CODES` are the nine language codes the library supports, of which `DEFAULT_LANG_CODE` (`a`, American English) is what this notebook loads, so a voice must start with that letter (`af_…`/`am_…`) and a British `bf_…` voice is rejected by design rather than silently mixed — the cell demonstrates exactly that rejection and records the pipeline's own error message as a finding. `SAMPLE_RATE` is the fixed 24 kHz output rate. The notebook never trims or alters the text; the library's 510-phoneme segment cap is applied inside the pipeline and is only visible afterwards through the returned `phonemes`.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MIN_SPEED': MIN_SPEED, 'MAX_SPEED': MAX_SPEED, 'SAMPLE_RATE': SAMPLE_RATE, 'LANG_CODES': LANG_CODES, 'DEFAULT_LANG_CODE': DEFAULT_LANG_CODE, 'DEFAULT_VOICE': DEFAULT_VOICE}
print(ceilings)
input_manifest = validate_inputs(text, VOICE, speed=SPEED, voices=pipe.voices, lang_code=pipe.lang_code, names=[sample_name])
# Demonstrate the cross-language rejection; the finding is recorded, not swallowed.
try:
    validate_inputs(text, 'bf_emma', speed=SPEED, voices=pipe.voices, lang_code=pipe.lang_code)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'wrong-language-voice-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/kokoro_tts_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2, ensure_ascii=False))

## 6. Synthesise, write the WAV, and read the output correctly

`synthesize(text, voice=…, speed=…)` returns `audio` (a 1-D float32 NumPy array at `sample_rate` 24000 Hz), `num_samples`, `duration_s`, `peak_amplitude`, `segments` (one `graphemes`/`phonemes` pair per newline-split segment), the `voice`, `lang_code` and `speed` used, `espeak_fallback`, the device, and the model identity. **The output is not bit-deterministic by default:** the vocoder adds Gaussian noise and a random initial phase to its harmonic excitation, so two unseeded calls differ at the sample level (the model card measured a maximum absolute difference of 0.093 between consecutive smoke calls of identical length) while durations and phonemes stay the same; the pipeline sets no seed, so this cell calls `torch.manual_seed(SEED)` immediately before synthesis — the card recorded bit-identical waveforms across two seeded calls on the same host. Seeding does not make the audio identical across devices, PyTorch builds or CPU kernels. The sanity checks below (right dtype, rate and shape, finite samples, peak within full scale, one segment per line) are falsifiable plumbing checks, not a quality result, and the model card's smoke observation (78,000 samples, 3.25 s, peak 0.342 on the same sentence, unseeded CPU) is quoted as one measurement on that host, not an expected value. The `phonemes` string is the only in-repository way to see whether the G2P dropped a word — with `espeak_fallback` `False`, out-of-dictionary words vanish silently. The WAV written to `outputs/` is 16-bit PCM at 24 kHz via the pinned `soundfile`; the inline player below appears only in an IPython front end.

In [ ]:
import time

torch.manual_seed(SEED)
started = time.perf_counter()
result = pipe.synthesize(text, voice=VOICE, speed=SPEED)
elapsed = time.perf_counter() - started
audio = result['audio']
checks = {
    'audio_is_float32_1d': isinstance(audio, np.ndarray) and audio.dtype == np.float32 and audio.ndim == 1,
    'sample_rate_matches_contract': result['sample_rate'] == SAMPLE_RATE,
    'all_samples_finite': bool(np.isfinite(audio).all()),
    'peak_within_full_scale': 0.0 < result['peak_amplitude'] <= 1.0,
    'one_segment_per_line': len(result['segments']) == len([line for line in text.splitlines() if line.strip()]),
    'duration_consistent': abs(result['duration_s'] - result['num_samples'] / SAMPLE_RATE) < 1e-6,
}
if not all(checks.values()):
    raise RuntimeError(f'synthesize output failed a sanity check: {checks}')
wav_path = 'outputs/kokoro_tts_sample.wav'
soundfile.write(wav_path, audio, result['sample_rate'], subtype='PCM_16')
wav_sha256 = hashlib.sha256(Path(wav_path).read_bytes()).hexdigest()
print({key: value for key, value in result.items() if key not in ('audio', 'segments')})
print({'seconds': round(elapsed, 3), 'audio_seconds_per_wall_second': round(result['duration_s'] / elapsed, 2), 'rms': round(float(np.sqrt(np.mean(np.square(audio)))), 4), 'checks': checks})
for index, segment in enumerate(result['segments']):
    print(f"segment {index}: graphemes={segment['graphemes'][:80]!r}")
    print(f"segment {index}: phonemes ={segment['phonemes'][:80]!r}")
print({'wav': wav_path, 'wav_sha256': wav_sha256, 'subtype': 'PCM_16'})
try:
    from IPython.display import Audio, display
    display(Audio(audio, rate=result['sample_rate']))
except ImportError:
    print('inline player unavailable outside an IPython front end; open the WAV file instead')

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report — even, as here, when nothing is measurable. The repository ships **no metric helper and reports no performance measure**: speech quality has no ground truth to compare against, so the verdict is always `not-measurable` and the report states what would make the task measurable — Mean Opinion Score ratings from human listeners for naturalness, or an independent speech recogniser to re-transcribe the waveform and compute word error rate against the input text for intelligibility (for example the `whisper-asr-pipeline` sibling), over a reference sentence set and with the judge named. Supplying a reference does not change the verdict, because no metric helper exists to score it; the helper records that in `reason` rather than inventing a number. The run-level facts it carries — segment count and duration — are observations, not scores. The report is written to `outputs/kokoro_tts_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, sample_kind=sample_kind)
with open('outputs/kokoro_tts_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2, ensure_ascii=False))
if report['verdict'] == 'not-measurable':
    print('No metric is reported: speech has no intrinsic ground truth, the sample has no reference recording, and the repository ships no metric helper; MOS needs human listeners and intelligibility needs an external ASR judge.')

## 8. Export outputs and provenance

Three files are written under `outputs/` beside the input manifest: the WAV from Section 6, the evaluation report from Section 7, and one JSON record with the WAV path and its SHA-256 (so the audio can be tied to this record), the run-level facts (`num_samples`, `duration_s`, `peak_amplitude`, RMS, wall time), the per-segment graphemes and phonemes, the request (`voice`, `lang_code`, `speed`, `seed`), `espeak_fallback`, the sanity checks, the ceilings in force, the input manifest, the evaluation report, the sample identity and text digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, the weight format and loader trust facts, the verified snapshot summary, and the runtime identity (Python, `torch`, `kokoro`, `misaki`, `soundfile`, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
payload = {
    'wav': {'path': wav_path, 'sha256': wav_sha256, 'subtype': 'PCM_16', 'sample_rate': result['sample_rate']},
    'audio': {'num_samples': result['num_samples'], 'duration_s': result['duration_s'], 'peak_amplitude': result['peak_amplitude'], 'rms': float(np.sqrt(np.mean(np.square(audio))))},
    'segments': result['segments'],
    'request': {'voice': result['voice'], 'lang_code': result['lang_code'], 'speed': result['speed'], 'seed': SEED},
    'espeak_fallback': result['espeak_fallback'],
    'sanity_checks': checks,
    'ceilings': ceilings,
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sample': {'name': sample_name, 'kind': sample_kind, 'text': text, 'text_sha256': text_sha256},
    'seconds': round(elapsed, 3),
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'weight_format': WEIGHT_FORMAT,
    'loader_weights_only': LOADER_WEIGHTS_ONLY,
    'snapshot': {'path': snapshot['path'], 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'voices': len(pipe.voices)},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'kokoro': kokoro.__version__,
        'misaki': misaki.__version__,
        'soundfile': soundfile.__version__,
        'device': pipe.device,
        'dtype': 'float32',
    },
}
with open('outputs/kokoro_tts_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The WAV is a synthetic rendering of the input text in one of 54 named synthetic voices: it is not a recording of any person, carries no quality score, and its only in-repository checks are structural (rate, dtype, finite samples, peak within full scale) plus the phoneme string that shows what the G2P actually voiced. On the synthetic sample the audio is plumbing evidence only; the evaluation report is `not-measurable` because no metric can be computed without an external judge, and a real evaluation needs MOS ratings from listeners or an ASR round-trip WER over a reference sentence set, with the judge named. The seed controls the vocoder noise on one host and build; it does not promise identical audio across devices. Out-of-dictionary words depend on the espeak-ng fallback and are dropped when it is absent; segments above 510 phonemes are truncated by the library; non-English `lang_code`s and non-English quality are not exercised here; the pipeline exposes no cloning, mixing, timestamps or emotion control. The checkpoint and voice packs are pickles loaded through the weights-only loader after digest verification — an operator who bypasses `verify_snapshot` and loads an unverified file takes on arbitrary-code-execution risk.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model snapshot including all voice packs, validate the demonstrated request against the enforced ceilings, execute the public pipeline path, write a playable WAV, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, naturalness or intelligibility on any audience, safety for high-consequence read-outs, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/kokoro-82m/` (or the affected `voices/*.pt`) and rerun Section 3. A `ValueError` naming `MAX_TEXT_CHARS`, the speed range or the voice in Section 5: fix the form parameter or shorten the BYOD file and rerun from Section 4. `espeak_fallback: False` in Section 3 with words missing from the `phonemes` string in Section 6: the bundled espeak-ng library did not bind on this host — restrict the text to dictionary words or install `espeak-ng` system-wide and reload. A long pause in Section 3 on first use: `misaki` is downloading the spaCy `en_core_web_sm` model.

**Next experiments.** Change `VOICE` to another `af_`/`am_` pack and compare the renderings of the same sentence; set `SPEED` to 0.8 and 1.5 and compare `duration_s`; run the same seed twice and diff the two WAV digests to see the seeded determinism on your host; upload a paragraph with names and numbers via `USE_BYOD` and inspect the `phonemes` string for dropped words; re-transcribe the WAV with the `whisper-asr-pipeline` sibling and compute WER against the input as the first step towards the intelligibility number the evaluation report asks for. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/kokoro-tts-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/kokoro-tts-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/kokoro-tts-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/hexgrad/Kokoro-82M
- Upstream code: https://github.com/hexgrad/kokoro (G2P: https://github.com/hexgrad/misaki)
- StyleTTS 2 (Li et al., 2023): https://arxiv.org/abs/2306.07691
- iSTFTNet (Kaneko et al., 2022): https://arxiv.org/abs/2203.02395